# Kiểm tra toàn bộ ảnh trong thư mục

Notebook này sẽ:

- quét **tất cả ảnh** trong một thư mục gốc và các thư mục con
- thống kê **tổng số ảnh**
- thống kê **kích thước ảnh** `(width x height)`
- phân loại **bao nhiêu ảnh cho từng kích thước**
- thống kê **ảnh màu** và **ảnh đen trắng**
- xuất kết quả chi tiết ra file CSV

## Cách dùng
1. Sửa biến `DATASET_DIR` ở ô bên dưới thành đường dẫn thư mục ảnh của bạn.
2. Chạy lần lượt các ô.
3. Xem bảng thống kê và file CSV được tạo ra.


In [ ]:
# Nếu thiếu thư viện, bỏ comment dòng dưới rồi chạy:
# !pip install pillow pandas

from pathlib import Path
from collections import Counter
from PIL import Image, ImageStat
import pandas as pd
import os

In [ ]:
# =========================
# CẤU HÌNH ĐƯỜNG DẪN DỮ LIỆU
# =========================
DATASET_DIR = r"./data"   # <-- sửa lại thành thư mục ảnh của bạn

# Định dạng ảnh cần quét
IMAGE_EXTS = {".jpg", ".jpeg", ".png", ".bmp", ".gif", ".tif", ".tiff", ".webp"}

In [ ]:
def is_image_file(path: Path) -> bool:
    return path.suffix.lower() in IMAGE_EXTS

def classify_color_type(img: Image.Image) -> str:
    """
    Trả về:
    - 'grayscale' nếu ảnh đen trắng / xám
    - 'color' nếu là ảnh màu
    """
    # Ảnh mode '1' hoặc 'L' chắc chắn là đen trắng/xám
    if img.mode in ("1", "L", "LA"):
        return "grayscale"

    # Với ảnh palette, chuyển sang RGB để kiểm tra
    if img.mode == "P":
        img = img.convert("RGB")

    # Nếu có alpha, bỏ alpha
    if img.mode in ("RGBA", "CMYK"):
        img = img.convert("RGB")

    # Nếu sau chuyển đổi vẫn là ảnh 1 kênh
    if img.mode not in ("RGB",):
        img = img.convert("RGB")

    # Ảnh được xem là grayscale nếu R == G == B trên toàn ảnh
    r, g, b = img.split()
    if list(r.getdata()) == list(g.getdata()) == list(b.getdata()):
        return "grayscale"
    return "color"

def scan_images(root_dir: str):
    root = Path(root_dir)
    if not root.exists():
        raise FileNotFoundError(f"Không tìm thấy thư mục: {root.resolve()}")

    records = []
    errors = []

    for path in root.rglob("*"):
        if path.is_file() and is_image_file(path):
            try:
                with Image.open(path) as img:
                    width, height = img.size
                    color_type = classify_color_type(img)
                    records.append({
                        "file_name": path.name,
                        "file_path": str(path.resolve()),
                        "folder": str(path.parent.resolve()),
                        "extension": path.suffix.lower(),
                        "width": width,
                        "height": height,
                        "size_label": f"{width}x{height}",
                        "mode": img.mode,
                        "color_type": color_type
                    })
            except Exception as e:
                errors.append({
                    "file_path": str(path.resolve()),
                    "error": str(e)
                })
    return pd.DataFrame(records), pd.DataFrame(errors)

In [ ]:
df, df_errors = scan_images(DATASET_DIR)

print(f"Tổng số ảnh đọc được: {len(df):,}")
print(f"Số file lỗi/không đọc được: {len(df_errors):,}")

df.head()

In [ ]:
# Thống kê tổng quan
total_images = len(df)
color_count = int((df["color_type"] == "color").sum()) if total_images else 0
gray_count = int((df["color_type"] == "grayscale").sum()) if total_images else 0
unique_sizes = df["size_label"].nunique() if total_images else 0

summary = pd.DataFrame([
    {"metric": "Tổng số ảnh", "value": total_images},
    {"metric": "Số kích thước khác nhau", "value": unique_sizes},
    {"metric": "Ảnh màu", "value": color_count},
    {"metric": "Ảnh đen trắng/xám", "value": gray_count},
])

summary

In [ ]:
# Phân loại số lượng ảnh theo từng kích thước
size_stats = (
    df.groupby(["width", "height", "size_label"], as_index=False)
      .size()
      .rename(columns={"size": "count"})
      .sort_values(["count", "width", "height"], ascending=[False, True, True])
      .reset_index(drop=True)
)

print(f"Có {len(size_stats):,} nhóm kích thước ảnh.")
size_stats.head(20)

In [ ]:
# Thống kê riêng ảnh màu / đen trắng theo từng kích thước
size_color_stats = (
    df.groupby(["size_label", "color_type"], as_index=False)
      .size()
      .rename(columns={"size": "count"})
      .sort_values(["size_label", "color_type"])
      .reset_index(drop=True)
)

size_color_stats.head(30)

In [ ]:
# Top kích thước phổ biến nhất
top_sizes = size_stats.head(20)
top_sizes

In [ ]:
# Các file lỗi (nếu có)
df_errors

In [ ]:
# Xuất kết quả ra CSV
output_dir = Path("./output")
output_dir.mkdir(exist_ok=True)

detail_csv = output_dir / "image_details.csv"
summary_csv = output_dir / "image_summary.csv"
size_csv = output_dir / "image_size_stats.csv"
size_color_csv = output_dir / "image_size_color_stats.csv"
error_csv = output_dir / "image_errors.csv"

df.to_csv(detail_csv, index=False, encoding="utf-8-sig")
summary.to_csv(summary_csv, index=False, encoding="utf-8-sig")
size_stats.to_csv(size_csv, index=False, encoding="utf-8-sig")
size_color_stats.to_csv(size_color_csv, index=False, encoding="utf-8-sig")
df_errors.to_csv(error_csv, index=False, encoding="utf-8-sig")

print("Đã xuất file:")
print("-", detail_csv.resolve())
print("-", summary_csv.resolve())
print("-", size_csv.resolve())
print("-", size_color_csv.resolve())
print("-", error_csv.resolve())

## Ghi chú về phân loại màu / đen trắng
Notebook này phân loại:

- **grayscale**: ảnh có các kênh màu giống nhau trên toàn ảnh, hoặc mode ảnh là `L`, `LA`, `1`
- **color**: ảnh có khác biệt giữa các kênh màu

Điều này phù hợp cho phần lớn dataset ảnh thực tế.
